# MiniGPT from Scratch — Colab demo

This notebook is a top-to-bottom reproduction walkthrough. Set the two clearly marked public URLs in the next cell; no other manual setup is required. It uses Colab's existing PyTorch installation rather than replacing it.

In [ ]:
# Replace GITHUB_URL with the public repository URL before sharing this notebook.
# Set CHECKPOINT_URL to a direct public download URL for minigpt_final.pt.
GITHUB_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPOSITORY.git"
CHECKPOINT_URL = ""  # e.g., a GitHub Release direct-download URL

if "YOUR_GITHUB_USERNAME" in GITHUB_URL or not CHECKPOINT_URL:
    raise ValueError("Set GITHUB_URL and CHECKPOINT_URL in this cell before Run all.")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("transformer-gpt-assignment")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

In [ ]:
# Colab already supplies torch. Install the remaining requirements without
# replacing the runtime-selected CPU/CUDA torch build.
requirements = [line.strip() for line in Path("requirements.txt").read_text().splitlines()]
dev_requirements = [line.strip() for line in Path("requirements-dev.txt").read_text().splitlines()]
packages = [line for line in requirements + dev_requirements if line and not line.startswith("#") and line.lower() != "torch"]
subprocess.run(["python", "-m", "pip", "install", "-q", *packages], check=True)
import torch
print("Using Colab torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())

In [ ]:
# Download Tiny Shakespeare if it is absent; the script tries Karpathy then a public HF mirror.
if not Path("data/input.txt").exists():
    subprocess.run(["python", "data/download_tiny_shakespeare.py"], check=True)

checkpoint_path = Path("checkpoints/minigpt_final.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint_path.exists():
    import urllib.request
    urllib.request.urlretrieve(CHECKPOINT_URL, checkpoint_path)
print("Checkpoint:", checkpoint_path, f"({checkpoint_path.stat().st_size / 2**20:.3f} MiB)")

In [ ]:
# Unit tests include attention, masking, shapes, dataset shift, and the overfit gate.
subprocess.run(["python", "-m", "pytest", "-q"], check=True)

In [ ]:
# Fresh-process inference from three fixed prompts.
for seed, prompt in enumerate(("ROMEO:", "KING HENRY:", "O Juliet,"), start=900):
    print(f"\n--- seed={seed}; prompt={prompt!r} ---")
    subprocess.run([
        "python", "-m", "src.generate", "--prompt", prompt,
        "--max-new-tokens", "80", "--temperature", "0.8", "--top-k", "40",
        "--seed", str(seed), "--checkpoint", "checkpoints/minigpt_final.pt",
    ], check=True)

In [ ]:
subprocess.run(["python", "-m", "experiments.hand_calculation"], check=True)

import torch
from IPython.display import Image, display
from src.attention import MultiHeadSelfAttention

display(Image(filename="outputs/figures/fig2_causal_mask.png"))
torch.manual_seed(7)
demo_attention = MultiHeadSelfAttention(d_model=8, n_heads=2, dropout=0.0).eval()
trace, _ = demo_attention.shape_trace(torch.randn(1, 3, 8), causal=True)
print(trace)
subprocess.run(["python", "-m", "experiments.multihead_shape_trace"], check=True)
display(Image(filename="outputs/figures/fig3_multihead_shape_trace.png"))

In [ ]:
# Optional learning exercise; left off by default so Run all does not retrain.
RUN_OPTIONAL_SHORT_TRAINING = False
if RUN_OPTIONAL_SHORT_TRAINING:
    import yaml
    config = yaml.safe_load(Path("configs/tiny_gpt.yaml").read_text())
    config.update({"max_steps": 200, "eval_interval": 100, "batch_size": 32, "checkpoint_dir": "checkpoints/colab_short"})
    Path("configs/colab_short.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
    subprocess.run(["python", "-m", "src.train", "--config", "configs/colab_short.yaml"], check=True)
else:
    print("Optional 200-step training is disabled. Set RUN_OPTIONAL_SHORT_TRAINING=True to run it.")

In [ ]:
# Optional ablation reproduction. The submitted study uses a measured 8,000-step budget per run.
RUN_OPTIONAL_ABLATIONS = False
if RUN_OPTIONAL_ABLATIONS:
    subprocess.run(["python", "experiments/ablation.py"], check=True)
else:
    display(Image(filename="outputs/figures/fig9_ablations.png"))
    print("Optional CUDA ablation rerun is disabled; the saved measured Figure 9 is shown instead.")